In [3]:
import pandas as pd
import numpy as np
import random

In [4]:
# Reproducibility 
random.seed(42)
np.random.seed(42)

In [6]:
STATES = [
    "Delhi", "Karnataka", "Gujarat", "Rajasthan", "Tamil Nadu",
    "Telangana", "Maharashtra", "Uttar Pradesh", "West Bengal",
    "Kerala", "Punjab", "Madhya Pradesh",
]

OPERATORS = ["Vi", "Jio", "Airtel", "Others", "BSNL"]

GENDER_CLEAN    = ["Male", "Female", "Other"]
PLANTYPE_CLEAN  = ["Prepaid", "Postpaid"]
YESNO_CLEAN     = ["Yes", "No"]

GENDER_DIRTY   = ["Male", "Female", "Other", "male", "female", "M", "F",
                   "MALE", "FEMALE", None]
PLANTYPE_DIRTY = ["Prepaid", "Postpaid", "prepaid", "POSTPAID", "postpaid", None]
YESNO_DIRTY    = ["Yes", "No", "yes", "no", "YES", "NO", "1", "0", None]
STATE_DIRTY    = STATES + [None, "N/A", "Unknown"]
OPERATOR_DIRTY = OPERATORS + [None, ""]

In [7]:
def new_customer_id(used: set) -> str:
    """Return a unique CUSTXXXXX id not already in `used`."""
    while True:
        num = random.randint(1, 99999)
        cid = f"CUST{num:05d}"
        if cid not in used:
            used.add(cid)
            return cid


def make_clean_row(cid: str) -> dict:
    """
    Generate one realistic, fully clean customer row.
    All values are within valid ranges and consistently formatted.
    """
    gender   = random.choice(GENDER_CLEAN)
    plantype = random.choice(PLANTYPE_CLEAN)

    # Churn probability influenced by satisfaction & complaints
    satisfaction   = random.randint(1, 5)
    net_complaints = random.randint(0, 10)
    churn_prob     = 0.10 + (0.08 * (5 - satisfaction)) + (0.03 * net_complaints)
    churn          = "Yes" if random.random() < min(churn_prob, 0.90) else "No"
    port_request   = "Yes" if churn == "Yes" and random.random() < 0.40 else "No"

    return {
        "CustomerID"           : cid,
        "Age"                  : random.randint(18, 70),
        "Gender"               : gender,
        "State"                : random.choice(STATES),
        "Operator"             : random.choice(OPERATORS),
        "SIMTenureMonths"      : random.randint(1, 120),
        "RechargeAmount"       : random.randint(99, 2999),
        "RechargeFrequencyDays": random.randint(14, 90),
        "DataUsageGB"          : round(random.uniform(0.5, 100.0), 2),
        "CallMinutes"          : random.randint(20, 3000),
        "SMSUsage"             : random.randint(0, 350),
        "PlanType"             : plantype,
        "NetworkComplaints"    : net_complaints,
        "DroppedCalls"         : random.randint(0, 50),
        "LateRechargeDays"     : random.randint(0, 30),
        "CustomerSupportCalls" : random.randint(0, 8),
        "SatisfactionScore"    : satisfaction,
        "PortRequest"          : port_request,
        "Churn"                : churn,
    }


def make_dirty_row(cid: str) -> dict:
    """
    Generate one dirty customer row with multiple quality issues:
    nulls, wrong casing, out-of-range numbers, bad Yes/No formats.
    """
    return {
        "CustomerID"           : cid,
        # Age: valid / negative / too high / null
        "Age"                  : random.choice([
                                     random.randint(18, 70),
                                     -5, 200, None,
                                 ]),
        "Gender"               : random.choice(GENDER_DIRTY),
        "State"                : random.choice(STATE_DIRTY),
        "Operator"             : random.choice(OPERATOR_DIRTY),
        # SIMTenureMonths: valid / negative / null
        "SIMTenureMonths"      : random.choice([
                                     random.randint(1, 120),
                                     -1, None,
                                 ]),
        # RechargeAmount: valid / negative / null
        "RechargeAmount"       : random.choice([
                                     random.randint(99, 2999),
                                     -100, None,
                                 ]),
        "RechargeFrequencyDays": random.choice([random.randint(14, 90), None]),
        "DataUsageGB"          : random.choice([
                                     round(random.uniform(0.5, 100.0), 2),
                                     None,
                                 ]),
        "CallMinutes"          : random.choice([random.randint(20, 3000), None]),
        "SMSUsage"             : random.choice([random.randint(0, 350), None]),
        "PlanType"             : random.choice(PLANTYPE_DIRTY),
        "NetworkComplaints"    : random.choice([random.randint(0, 10), None]),
        "DroppedCalls"         : random.choice([random.randint(0, 50), None]),
        "LateRechargeDays"     : random.choice([random.randint(0, 30), None]),
        "CustomerSupportCalls" : random.choice([random.randint(0, 8), None]),
        # SatisfactionScore: valid / out-of-range (0 or 6) / null
        "SatisfactionScore"    : random.choice([
                                     random.randint(1, 5),
                                     0, 6, None,
                                 ]),
        "PortRequest"          : random.choice(YESNO_DIRTY),
        "Churn"                : random.choice(YESNO_DIRTY),
    }


In [9]:
used_ids   = set()
clean_rows = [make_clean_row(new_customer_id(used_ids)) for _ in range(10_000)]
clean_df   = pd.DataFrame(clean_rows)

In [10]:
dirty_rows = [make_dirty_row(new_customer_id(used_ids)) for _ in range(2_000)]
dirty_df   = pd.DataFrame(dirty_rows)

dup_dirty = dirty_df.sample(300, replace=False, random_state=42)
dirty_df  = pd.concat([dirty_df, dup_dirty], ignore_index=True)

dup_clean = clean_df.sample(200, replace=False, random_state=42)
dirty_df  = pd.concat([dirty_df, dup_clean], ignore_index=True)

In [11]:
insert_positions = sorted(random.sample(range(len(clean_df)), k=len(dirty_df)))

parts = []
prev  = 0
for pos, (_, dirty_row) in zip(insert_positions, dirty_df.iterrows()):
    parts.append(clean_df.iloc[prev:pos])
    parts.append(pd.DataFrame([dirty_row.to_dict()]))
    prev = pos
parts.append(clean_df.iloc[prev:])

combined = pd.concat(parts, ignore_index=True)

In [14]:
age_bad = combined[combined["Age"].notna() & ((combined["Age"] < 18) | (combined["Age"] > 99))]
score_bad = combined[combined["SatisfactionScore"].notna() &
                     ((combined["SatisfactionScore"] < 1) | (combined["SatisfactionScore"] > 5))]

In [15]:
combined.to_csv("data.csv", index=False)

In [ ]:
|